In [1]:
import pandas as pd
import numpy as np
import os
import arff

In [2]:
# ============================================================
# FOLDERS
# ============================================================

BINARY_PATH = "../data/binary_target/"
THREE_PATH = "../data/three_class_target/"
MULTI_PATH = "../data/multi_class_target/"

MERGED_PATH = "../data/merged/"
HIER_PATH = "../data/datasets_hierarchical/"

os.makedirs(MERGED_PATH, exist_ok=True)
os.makedirs(HIER_PATH, exist_ok=True)

# ============================================================
# FUNCTION: MERGE CSV
# ============================================================

def merge_csv(folder):

    files = sorted([f for f in os.listdir(folder) if f.endswith(".csv")])
    dfs = []

    for f in files:
        df = pd.read_csv(os.path.join(folder, f))
        dfs.append(df)
        print("Loaded", f, df.shape)

    merged = pd.concat(dfs, ignore_index=True)

    print("Merged shape:", merged.shape)

    return merged


# ============================================================
# FUNCTION: MERGE ARFF
# ============================================================

def merge_arff(folder):

    files = [f for f in os.listdir(folder) if f.endswith(".arff")]
    dfs = []

    for f in files:

        with open(os.path.join(folder, f)) as file:
            data = arff.load(file)

        df = pd.DataFrame(
            data["data"],
            columns=[a[0] for a in data["attributes"]]
        )

        dfs.append(df)

        print("Loaded", f, df.shape)

    merged = pd.concat(dfs, ignore_index=True)

    print("Merged shape:", merged.shape)

    return merged


# ============================================================
# FUNCTION: CLEAN DATA
# ============================================================

def clean_dataset(df):

    df = df.copy()

    z_cols = ["R1-PA:Z","R2-PA:Z","R3-PA:Z","R4-PA:Z"]

    # create flags
    for col in z_cols:
        if col in df.columns:
            df[col + "_inf_flag"] = np.isinf(df[col]).astype(int)

    # replace inf
    df = df.replace([np.inf, -np.inf], np.nan)

    # fill NaN
    df = df.fillna(df.median(numeric_only=True))

    # columns to exclude
    exclude_cols = [
        "marker",
        "Scenario_ID",
        "target_binary",
        "target_three_class"
    ]

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    numeric_cols = [
        c for c in numeric_cols
        if c not in exclude_cols
        and "log" not in c.lower()
        and "flag" not in c.lower()
        and not c.endswith(":F")
        and not c.endswith(":DF")
    ]

    # IQR outlier capping
    for col in numeric_cols:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df[col] = df[col].clip(lower, upper)

    print("Outlier capping applied")
    print("Dataset shape:", df.shape)

    return df

In [3]:
# ============================================================
# BINARY DATASET
# ============================================================

binary_raw = merge_csv(BINARY_PATH)

binary_raw.to_csv(
    MERGED_PATH + "binary_dataset.csv",
    index=False
)

binary_clean = clean_dataset(binary_raw)

binary_clean.to_csv(
    MERGED_PATH + "binary_dataset_clean_FULL.csv",
    index=False
)

print("Binary datasets saved")

Loaded data1.csv (4966, 129)
Loaded data10.csv (5569, 129)
Loaded data11.csv (5251, 129)
Loaded data12.csv (5224, 129)
Loaded data13.csv (5271, 129)
Loaded data14.csv (5115, 129)
Loaded data15.csv (5276, 129)
Loaded data2.csv (5069, 129)
Loaded data3.csv (5415, 129)
Loaded data4.csv (5202, 129)
Loaded data5.csv (5161, 129)
Loaded data6.csv (4967, 129)
Loaded data7.csv (5236, 129)
Loaded data8.csv (5315, 129)
Loaded data9.csv (5340, 129)
Merged shape: (78377, 129)
Outlier capping applied
Dataset shape: (78377, 133)
Binary datasets saved


In [4]:
# ============================================================
# THREE CLASS DATASET
# ============================================================

three_raw = merge_csv(THREE_PATH)

three_raw.to_csv(
    MERGED_PATH + "three_class_dataset.csv",
    index=False
)

three_clean = clean_dataset(three_raw)

three_clean.to_csv(
    MERGED_PATH + "three_class_dataset_clean_FULL.csv",
    index=False
)

print("Three-class datasets saved")

Loaded data1.csv (4966, 129)
Loaded data10.csv (5569, 129)
Loaded data11.csv (5251, 129)
Loaded data12.csv (5224, 129)
Loaded data13.csv (5271, 129)
Loaded data14.csv (5115, 129)
Loaded data15.csv (5276, 129)
Loaded data2.csv (5069, 129)
Loaded data3.csv (5415, 129)
Loaded data4.csv (5202, 129)
Loaded data5.csv (5161, 129)
Loaded data6.csv (4967, 129)
Loaded data7.csv (5236, 129)
Loaded data8.csv (5315, 129)
Loaded data9.csv (5340, 129)
Merged shape: (78377, 129)
Outlier capping applied
Dataset shape: (78377, 133)
Three-class datasets saved


In [5]:
# ============================================================
# MULTI CLASS DATASET
# ============================================================

multi_df = merge_arff(MULTI_PATH)

# ------------------------------------------------------------
# Clean column names
# ------------------------------------------------------------

multi_df.columns = multi_df.columns.str.strip()

print("Original dataset shape:", multi_df.shape)


# ------------------------------------------------------------
# SAVE RAW DATASET (for EDA)
# ------------------------------------------------------------

multi_df.to_csv(
    MERGED_PATH + "multi_class_dataset.csv",
    index=False
)

print("Saved RAW dataset for EDA")


# ------------------------------------------------------------
# 1. ENSURE NUMERIC FEATURES
# ------------------------------------------------------------

feature_cols = multi_df.columns.drop("marker")

multi_df[feature_cols] = multi_df[feature_cols].apply(
    pd.to_numeric,
    errors="coerce"
)


# ------------------------------------------------------------
# 2. DETECT IMPEDANCE INF VALUES
# ------------------------------------------------------------

Z_COLUMNS = [
    "R1-PA:Z",
    "R2-PA:Z",
    "R3-PA:Z",
    "R4-PA:Z"
]

print("\nChecking impedance INF values:")

flag_columns = {}

for col in Z_COLUMNS:

    if col in multi_df.columns:

        inf_mask = np.isinf(multi_df[col])

        print(f"{col}: {inf_mask.sum()} INF values")

        flag_columns[col + "_inf_flag"] = inf_mask.astype(int)


# add INF flag features
multi_df = pd.concat(
    [multi_df, pd.DataFrame(flag_columns)],
    axis=1
)


# ------------------------------------------------------------
# 3. REPLACE INF WITH NaN
# ------------------------------------------------------------

multi_df = multi_df.replace(
    [np.inf, -np.inf],
    np.nan
)


# ------------------------------------------------------------
# 4. FILL NaN VALUES (SAFE)
# ------------------------------------------------------------

# continuous features only
numeric_cols = multi_df.select_dtypes(
    include=[np.number]
).columns

numeric_cols = [
    c for c in numeric_cols
    if (
        "log" not in c.lower()
        and "flag" not in c.lower()
        and c != "marker"
    )
]

multi_df[numeric_cols] = multi_df[numeric_cols].fillna(
    multi_df[numeric_cols].median()
)


# ------------------------------------------------------------
# 5. VALIDATE DATASET
# ------------------------------------------------------------

print("\nDataset validation")

print("Shape:", multi_df.shape)

print(
    "Remaining NaN:",
    multi_df.isna().sum().sum()
)

print(
    "Remaining INF:",
    np.isinf(
        multi_df.select_dtypes(include=[np.number])
    ).sum().sum()
)


# ------------------------------------------------------------
# 5. CAP OUTLIERS (IQR)
# ------------------------------------------------------------

print("\nApplying outlier capping...")

cap_cols = [
    c for c in multi_df.select_dtypes(include=np.number).columns
    if "log" not in c.lower()
    and "flag" not in c.lower()
    and c != "marker"
    and not c.endswith(":F")
    and not c.endswith(":DF")
]

for col in cap_cols:

    Q1 = multi_df[col].quantile(0.25)
    Q3 = multi_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    multi_df[col] = multi_df[col].clip(lower, upper)

print("Outlier capping complete")

# ------------------------------------------------------------
# 6. SAVE CLEAN DATASET
# ------------------------------------------------------------

multi_df.to_csv(
    MERGED_PATH + "multi_class_dataset_clean_FULL.csv",
    index=False
)

print("Saved CLEAN dataset for modeling")

Loaded data5 Sampled Scenarios.csv.arff (5161, 129)
Loaded data9 Sampled Scenarios.csv.arff (5340, 129)
Loaded data14 Sampled Scenarios.csv.arff (5115, 129)
Loaded data13 Sampled Scenarios.csv.arff (5271, 129)
Loaded data2 Sampled Scenarios.csv.arff (5069, 129)
Loaded data12 Sampled Scenarios.csv.arff (5224, 129)
Loaded data3 Sampled Scenarios.csv.arff (5415, 129)
Loaded data4 Sampled Scenarios.csv.arff (5202, 129)
Loaded data8 Sampled Scenarios.csv.arff (5315, 129)
Loaded data15 Sampled Scenarios.csv.arff (5276, 129)
Loaded data1 Sampled Scenarios.csv.arff (4966, 129)
Loaded data10 Sampled Scenarios.csv.arff (5569, 129)
Loaded data6 Sampled Scenarios.csv.arff (4967, 129)
Loaded data7 Sampled Scenarios.csv.arff (5236, 129)
Loaded data11 Sampled Scenarios.csv.arff (5251, 129)
Merged shape: (78377, 129)
Original dataset shape: (78377, 129)
Saved RAW dataset for EDA

Checking impedance INF values:
R1-PA:Z: 2877 INF values
R2-PA:Z: 2607 INF values
R3-PA:Z: 2633 INF values
R4-PA:Z: 2789 INF

In [6]:
# 2. ENSURE NUMERIC FEATURES
# ------------------------------------------------------------

feature_cols = multi_df.columns.drop("marker")

multi_df[feature_cols] = multi_df[feature_cols].apply(
    pd.to_numeric, errors="coerce"
)

# ------------------------------------------------------------
# 3. DETECT IMPEDANCE INF VALUES
# ------------------------------------------------------------

Z_COLUMNS = [
    "R1-PA:Z",
    "R2-PA:Z",
    "R3-PA:Z",
    "R4-PA:Z"
]

print("\nChecking impedance INF values:")

flag_columns = {}

for col in Z_COLUMNS:

    inf_mask = np.isinf(multi_df[col])

    inf_count = inf_mask.sum()

    print(f"{col}: {inf_count} inf values")

    flag_columns[col + "_inf_flag"] = inf_mask.astype(int)

multi_df = pd.concat([multi_df, pd.DataFrame(flag_columns)], axis=1)

# ------------------------------------------------------------
# 4. REPLACE INF WITH NaN
# ------------------------------------------------------------

multi_df = multi_df.replace([np.inf, -np.inf], np.nan)

# ------------------------------------------------------------
# 5. FILL NaN VALUES
# ------------------------------------------------------------

multi_df = multi_df.fillna(multi_df.median(numeric_only=True))

# ------------------------------------------------------------
# 6. VALIDATE DATASET
# ------------------------------------------------------------

print("\nDataset validation")
print("After full cleaning:")

print("Shape:", multi_df.shape)

print("Remaining NaN:", multi_df.isna().sum().sum())

print(
    "Remaining INF:",
    np.isinf(
        multi_df.select_dtypes(include=[np.number])
    ).sum().sum()
)

# ------------------------------------------------------------
# 7. SAVE CLEAN DATASET
# ------------------------------------------------------------
OUTPUT = "../data/merged/multi_class_dataset_clean_FULL.csv"

multi_df.to_csv(OUTPUT, index=False)

print("Saved cleaned dataset:")
print(OUTPUT)


Checking impedance INF values:
R1-PA:Z: 0 inf values
R2-PA:Z: 0 inf values
R3-PA:Z: 0 inf values
R4-PA:Z: 0 inf values

Dataset validation
After full cleaning:
Shape: (78377, 137)
Remaining NaN: 0
Remaining INF: 0
Saved cleaned dataset:
../data/merged/multi_class_dataset_clean_FULL.csv


In [7]:
# ============================================================
# BUILD HIERARCHICAL DATASETS
# ============================================================

df = multi_df.copy()

import pandas as pd
import os

OUTPUT_DIR = "../data/datasets_hierarchical"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load fully cleaned dataset
df = pd.read_csv("../data/merged/multi_class_dataset_clean_FULL.csv")
print("Loaded FULL cleaned dataset:", df.shape)

FEATURE_COLUMNS = [c for c in df.columns if c != "marker"]

# ------------- Scenario groups ----------------
faults = list(range(1, 7))
maintenance = [13, 14]
normal = [41]

data_injection = list(range(7, 13))
remote_tripping = list(range(15, 21))
relay_setting = list(range(21, 31)) + list(range(35, 41))

all_non_attacks = faults + maintenance + normal
all_attacks = data_injection + remote_tripping + relay_setting

# ============================================================
# M1 — ATTACK vs NON-ATTACK (NO MARKER)
# ============================================================
df_m1 = df[FEATURE_COLUMNS + ["marker"]].copy()
df_m1["label"] = df_m1["marker"].apply(lambda m: 1 if m in all_attacks else 0)
df_m1["label_name"] = df_m1["label"].map({0: "Non-Attack", 1: "Attack"})

df_m1 = df_m1.drop(columns=["marker"])
df_m1.to_csv(f"{OUTPUT_DIR}/M1_Attack_vs_NonAttack_clean.csv", index=False)
print("Saved M1:", df_m1.shape)

# ============================================================
# M2 — NATURAL 3-CLASS (NO MARKER)
# ============================================================
df_m2 = df[df["marker"].isin(all_non_attacks)][FEATURE_COLUMNS + ["marker"]].copy()

def map_nat(m):
    if m in faults:
        return 0
    elif m in maintenance:
        return 1
    else:
        return 2

df_m2["label"] = df_m2["marker"].apply(map_nat)
df_m2["label_name"] = df_m2["label"].map({
    0: "Short-Circuit Fault",
    1: "Line Maintenance",
    2: "Normal Operation"
})

df_m2 = df_m2.drop(columns=["marker"])
df_m2.to_csv(f"{OUTPUT_DIR}/M2_Natural_3Class_clean.csv", index=False)
print("Saved M2:", df_m2.shape)

# ============================================================
# M3 — ATTACK 3-CLASS (NO MARKER)
# ============================================================
df_m3 = df[df["marker"].isin(all_attacks)][FEATURE_COLUMNS + ["marker"]].copy()

def map_attack(m):
    if m in data_injection:
        return 0
    elif m in remote_tripping:
        return 1
    else:
        return 2

df_m3["label"] = df_m3["marker"].apply(map_attack)
df_m3["label_name"] = df_m3["label"].map({
    0: "Data Injection",
    1: "Remote Tripping",
    2: "Relay Setting Change"
})

df_m3 = df_m3.drop(columns=["marker"])
df_m3.to_csv(f"{OUTPUT_DIR}/M3_Attack_3Class_clean.csv", index=False)
print("Saved M3:", df_m3.shape)

# ============================================================
# INTERNAL ATTACK LEVELS
# ============================================================

df_m4 = df[df["marker"].isin(data_injection)].copy()
df_m4["label"] = df_m4["marker"]

df_m4.to_csv(f"{OUTPUT_DIR}/M4_DataInjection_Internal_clean.csv", index=False)
print("Saved M4:", df_m4.shape)

df_m5 = df[df["marker"].isin(remote_tripping)].copy()
df_m5["label"] = df_m5["marker"]

df_m5.to_csv(f"{OUTPUT_DIR}/M5_RemoteTripping_Internal_clean.csv", index=False)
print("Saved M5:", df_m5.shape)

df_m6 = df[df["marker"].isin(relay_setting)].copy()
df_m6["label"] = df_m6["marker"]

df_m6.to_csv(f"{OUTPUT_DIR}/M6_RelaySetting_Internal_clean.csv", index=False)
print("Saved M6:", df_m6.shape)

print("\n🎉 All hierarchical datasets regenerated successfully!")


Loaded FULL cleaned dataset: (78377, 137)
Saved M1: (78377, 138)
Saved M2: (22714, 138)
Saved M3: (55663, 138)
Saved M4: (9582, 138)
Saved M5: (8737, 138)
Saved M6: (37344, 138)

🎉 All hierarchical datasets regenerated successfully!


In [8]:
import pandas as pd

BASE_PATH = "../data/datasets_hierarchical/"

datasets = {
    "M1": "M1_Attack_vs_NonAttack_clean.csv",
    "M2": "M2_Natural_3Class_clean.csv",
    "M3": "M3_Attack_3Class_clean.csv",
    "M4": "M4_DataInjection_Internal_clean.csv",
    "M5": "M5_RemoteTripping_Internal_clean.csv",
    "M6": "M6_RelaySetting_Internal_clean.csv"
}

feature_sets = {}

for name, file in datasets.items():

    print("\n====================================")
    print(f"Checking {name}")
    print("====================================")

    df = pd.read_csv(BASE_PATH + file)

    print("Shape:", df.shape)

    print("\nLast columns:")
    print(df.columns[-5:])

    # Feature separation (important for modelling)
    feature_cols = [c for c in df.columns if c not in ["label", "label_name"]]

    print("\nFeature count:", len(feature_cols))

    feature_sets[name] = set(feature_cols)

    # Unique labels
    if "label" in df.columns:
        print("\nUnique labels:")
        print(sorted(df["label"].unique()))

    # Class distribution
    if "label_name" in df.columns:

        print("\nClass counts:")
        print(df["label_name"].value_counts())

        print("\nClass percentage:")
        print((df["label_name"].value_counts(normalize=True)*100).round(2))

    else:

        print("\nInternal attack levels distribution:")
        print(df["label"].value_counts())

# ------------------------------------------------
# FEATURE CONSISTENCY CHECK (CRITICAL)
# ------------------------------------------------

print("\n\n====================================")
print("Feature Consistency Check")
print("====================================")

base_features = feature_sets["M1"]

for name in feature_sets:

    match = base_features == feature_sets[name]

    print(f"M1 vs {name} feature match:", match)


Checking M1
Shape: (78377, 138)

Last columns:
Index(['R2-PA:Z_inf_flag.1', 'R3-PA:Z_inf_flag.1', 'R4-PA:Z_inf_flag.1',
       'label', 'label_name'],
      dtype='object')

Feature count: 136

Unique labels:
[0, 1]

Class counts:
label_name
Attack        55663
Non-Attack    22714
Name: count, dtype: int64

Class percentage:
label_name
Attack        71.02
Non-Attack    28.98
Name: proportion, dtype: float64

Checking M2
Shape: (22714, 138)

Last columns:
Index(['R2-PA:Z_inf_flag.1', 'R3-PA:Z_inf_flag.1', 'R4-PA:Z_inf_flag.1',
       'label', 'label_name'],
      dtype='object')

Feature count: 136

Unique labels:
[0, 1, 2]

Class counts:
label_name
Short-Circuit Fault    15000
Normal Operation        4405
Line Maintenance        3309
Name: count, dtype: int64

Class percentage:
label_name
Short-Circuit Fault    66.04
Normal Operation       19.39
Line Maintenance       14.57
Name: proportion, dtype: float64

Checking M3
Shape: (55663, 138)

Last columns:
Index(['R2-PA:Z_inf_flag.1', 'R